In [ ]:
#Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/raw/paysim.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded successfully: {DATA_PATH.exists()}")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"   Columns : {df.columns.tolist()}")

In [ ]:
#Cell 2: Basic Inspection
#first 5 rows of the dataset
print("=" * 60)
print("SAMPLE ROWS")
print("=" * 60)
display(df.head())

#data types of every column
print("=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes.to_frame(name="dtype"))

#memory usage
print("\n" + "=" * 60)
print("MEMORY USAGE")
print("=" * 60)
mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"    Total memory used: {mem_mb:.2f} MB")

#quick statistical summary
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
display(df.describe())


In [ ]:
#Cell 3 : Missing Values & Duplicates
#missing values per column
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": df.isnull().mean() * 100
}).sort_values("missing_count", ascending=False)

display(missing)

#total missing across entire dataframe
total_missing = df.isnull().sum().sum()
print(f"\n  Total missing cells: {total_missing:,}")
print(f"  Dataset is {'CLEAN' if total_missing == 0 else 'HAS NULLS'}")

#duplicate rows
print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

num_dupes = df.duplicated().sum()
print(f"   Duplicate rows found: {num_dupes:,}")
print(f"   Dataset is {'CLEAN' if num_dupes == 0 else 'HAS DUPLICATES'}")

#sanity check: isFraud and isFlaggedFraud should only contain 0s and 1s
print("\n" + "=" * 60)
print("LABEL SANITY CHECK")
print("=" * 60)

for col in ["isFraud", "isFlaggedFraud"]:
    unique_vals = sorted(df[col].unique().tolist())
    print(f"   {col}: unique values = {unique_vals}")


In [ ]:
#Cell 4: Fraud Rate & Class Imbalance
#overall fraud rate
print("=" * 60)
print("OVERALL FRAUD RATE")
print("=" * 60)

fraud_count = df["isFraud"].sum()
legit_count = len(df) - fraud_count
fraud_rate_pct = df["isFraud"].mean() * 100

print(f"   Total transactions  : {len(df):>12,}")
print(f"   Fraudulent          : {fraud_count:>12,} ({fraud_rate_pct:.4f}%)")
print(f"   Legitimate          : {legit_count:>12,} ({100 - fraud_rate_pct:.4f}%)")
print(f"   Imbalance Ratio     : 1 fraud per {legit_count / fraud_count:.2f} legit transactions")

#isFlaggedFraud breakdown
print("\n" + "=" * 60)
print("isFlaggedFraud BREAKDOWN")
print("=" * 60)

flagged_count = df["isFlaggedFraud"].sum()
print(f"   Flagged as fraud    : {flagged_count:>12,}")
print(f"   isFraud=1 but NOT flagged : {(df['isFraud'] - df['isFlaggedFraud']).clip(lower=0).sum():,}")

#cross-tabulation: isFraud vs isFlaggedFraud
print("\n" + "=" * 60)
print("CROSS-TAB: isFraud vs isFlaggedFraud")
print("=" * 60)

crosstab = pd.crosstab(
    df["isFraud"],
    df["isFlaggedFraud"],
    rownames=["isFraud"],
    colnames=["isFlaggedFraud"],
    margins=True
)
display(crosstab)

#fraud breakdown by transaction type
print("\n" + "=" * 60)
print("FRAUD COUNT BY TRANSACTION TYPE")
print("=" * 60)

fraud_by_type = df.groupby("type")["isFraud"].agg(
    total_txns   = "count",
    fraud_txns   = "sum",
).assign(fraud_rate_pct = lambda x: x["fraud_txns"] / x["total_txns"] * 100)

fraud_by_type = fraud_by_type.sort_values("fraud_txns", ascending=False)

display(fraud_by_type)

In [ ]:
#Cell 5 : Transaction Type Distribution

#count and proportion of each transaction type
print("=" * 60)
print("TRANSACTION TYPE DISTRIBUTION")
print("=" * 60)

type_dist = df.groupby("type").agg(
    count        = ("type", "count"),
    fraud_count  = ("isFraud", "sum"),
    total_amount = ("amount", "sum"),
    mean_amount  = ("amount", "mean"),
).assign(
    pct_of_txns   = lambda x: x["count"] / x["count"].sum() * 100,
    pct_of_volume = lambda x: x["total_amount"] / x["total_amount"].sum() * 100,
).sort_values("count", ascending=False)                     # (8)

type_dist["total_amount"] = type_dist["total_amount"].map("{:,.2f}".format)
type_dist["mean_amount"]  = type_dist["mean_amount"].map("{:,.2f}".format)
type_dist["pct_of_txns"]  = type_dist["pct_of_txns"].map("{:.2f}%".format)
type_dist["pct_of_volume"]= type_dist["pct_of_volume"].map("{:.2f}%".format)

display(type_dist)

#value counts as a quick double-check
print("\n" + "=" * 60)
print("RAW VALUE COUNTS (sanity check)")
print("=" * 60)

vc = df["type"].value_counts()
display(vc.to_frame(name="count"))

# 5c) Fraud-eligible types only
print("\n" + "=" * 60)
print("FRAUD-ELIGIBLE TYPES (TRANSFER & CASH_OUT only)")
print("=" * 60)

fraud_eligible = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]
print(f"   Rows in TRANSFER + CASH_OUT : {len(fraud_eligible):,}")
print(f"   % of total dataset          : {len(fraud_eligible)/len(df)*100:.2f}%")
print(f"   Fraud within these rows     : {fraud_eligible['isFraud'].sum():,}")
print(f"   Fraud rate within these     : {fraud_eligible['isFraud'].mean()*100:.4f}%")


In [8]:
#Cell 6: Amount & Balance Distributions

# amount statistics split by fraud label
print("=" * 60)
print("AMOUNT STATISTICS: FRAUD vs LEGIT")
print("=" * 60)

amount_stats = df.groupby("isFraud")["amount"].describe()
amount_stats.index = amount_stats.index.map({0: "Legit", 1: "Fraud"})
display(amount_stats)

#amount percentiles for fraud transactions
print("\n" + "=" * 60)
print("FRAUD AMOUNT PERCENTILES")
print("=" * 60)

fraud_amounts = df.loc[df["isFraud"] == 1, "amount"]
percentiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
pct_df = fraud_amounts.quantile(percentiles).to_frame(name="amount")
pct_df.index = [f"{int(p*100)}th" for p in percentiles]
pct_df["amount"] = pct_df["amount"].map("{:,.2f}".format)
display(pct_df)

#balance error check — the key PaySim insight
print("\n" + "=" * 60)
print("BALANCE CONSISTENCY CHECK (Fraud vs Legit)")
print("=" * 60)

df["orig_balance_diff"] = df["newbalanceOrig"] - (df["oldbalanceOrg"] - df["amount"])
df["dest_balance_diff"] = df["newbalanceDest"] - (df["oldbalanceDest"] + df["amount"])

balance_check = df.groupby("isFraud")[["orig_balance_diff", "dest_balance_diff"]].agg(
    ["mean", "std", "min", "max"]
)
balance_check.index = balance_check.index.map({0: "Legit", 1: "Fraud"})
display(balance_check)

#how often do fraud transactions have zero new balance for origin?
print("\n" + "=" * 60)
print("ZERO BALANCE AFTER TRANSACTION (Origin Account)")
print("=" * 60)

for label, label_name in [(0, "Legit"), (1, "Fraud")]:
    subset = df[df["isFraud"] == label]
    zero_bal = (subset["newbalanceOrig"] == 0).sum()
    zero_pct = zero_bal / len(subset) * 100
    print(f"   {label_name:5s} → newbalanceOrig == 0 : {zero_bal:>8,}  ({zero_pct:.2f}%)")


AMOUNT STATISTICS: FRAUD vs LEGIT


,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
Legit,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
Fraud,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00



FRAUD AMOUNT PERCENTILES


,amount
10th,"37,720.99"
25th,"127,091.33"
50th,"441,423.44"
75th,"1,517,771.48"
90th,"4,521,723.51"
95th,"8,006,429.04"
99th,"10,000,000.00"
100th,"10,000,000.00"



BALANCE CONSISTENCY CHECK (Fraud vs Legit)


orig_balance_diff                                            \
                     mean            std           min          max   
isFraud                                                               
Legit       201338.558109  606928.890826 -1.000000e-02  92445516.64   
Fraud        10692.325265  265146.131130 -3.725290e-09  10000000.00   

        dest_balance_diff                                          
                     mean           std          min          max  
isFraud                                                            
Legit       -54692.231734  4.360026e+05 -13191233.98  75885725.63  
Fraud      -732509.301069  1.867748e+06 -10000000.00   8875516.29


ZERO BALANCE AFTER TRANSACTION (Origin Account)
   Legit → newbalanceOrig == 0 : 3,601,513  (56.68%)
   Fraud → newbalanceOrig == 0 :    8,053  (98.05%)


In [ ]:
# Cell 7 : Time (Step) Analysis

# Transactions per step
txns_per_step = df.groupby("step").agg(
    total_txns  = ("isFraud", "count"),
    fraud_txns  = ("isFraud", "sum"),
    total_amount= ("amount", "sum"),
).assign(
    fraud_rate_pct = lambda x: x["fraud_txns"] / x["total_txns"] * 100
)

print("=" * 60)
print("TRANSACTIONS PER STEP — HEAD & TAIL")
print("=" * 60)
display(txns_per_step.head(10))
display(txns_per_step.tail(10))

# Overall step range and simulation window
print("\n" + "=" * 60)
print("STEP RANGE")
print("=" * 60)
print(f"   Min step : {df['step'].min()}")
print(f"   Max step : {df['step'].max()}")
print(f"   Unique steps : {df['step'].nunique()}")
print(f"   Equivalent days : {df['step'].max() / 24:.1f} days")

# Peak fraud hours — which steps have the most fraud?
print("\n" + "=" * 60)
print("TOP 10 STEPS BY FRAUD COUNT")
print("=" * 60)
top_fraud_steps = txns_per_step.sort_values("fraud_txns", ascending=False).head(10)
display(top_fraud_steps)

# Fraud distribution across time thirds (early / mid / late month)
print("\n" + "=" * 60)
print("FRAUD ACROSS TIME THIRDS (Early / Mid / Late)")
print("=" * 60)
max_step = df["step"].max()
df["time_period"] = pd.cut(
    df["step"],
    bins=[0, max_step // 3, (max_step // 3) * 2, max_step],
    labels=["Early", "Mid", "Late"],
    include_lowest=True
)

period_stats = df.groupby("time_period", observed=True).agg(
    total_txns = ("isFraud", "count"),
    fraud_txns = ("isFraud", "sum"),
).assign(
    fraud_rate_pct = lambda x: x["fraud_txns"] / x["total_txns"] * 100
)
display(period_stats)


In [12]:
# Cell 8 : Save EDA Summary & Wrap Up Day 2

from pathlib import Path
import json
from datetime import datetime

# Recompute key values cleanly for the summary
total_rows       = len(df)
fraud_count      = int(df["isFraud"].sum())
legit_count      = total_rows - fraud_count
fraud_rate_pct   = df["isFraud"].mean() * 100
flagged_count    = int(df["isFlaggedFraud"].sum())
fraud_types      = df[df["isFraud"] == 1]["type"].unique().tolist()
zero_bal_fraud   = int((df[df["isFraud"] == 1]["newbalanceOrig"] == 0).sum())
zero_bal_pct     = zero_bal_fraud / fraud_count * 100
fraud_mean_amt   = df[df["isFraud"] == 1]["amount"].mean()
legit_mean_amt   = df[df["isFraud"] == 0]["amount"].mean()

summary = {
    "project"        : "NuroGuard — USM VHack 2026",
    "notebook"       : "01_eda.ipynb",
    "dataset"        : "PaySim (data/raw/paysim.csv)",
    "generated_at"   : datetime.now().isoformat(),
    "dataset_shape"  : {"rows": total_rows, "columns": df.shape[1]},
    "class_balance"  : {
        "fraud_count"     : fraud_count,
        "legit_count"     : legit_count,
        "fraud_rate_pct"  : round(fraud_rate_pct, 4),
        "imbalance_ratio" : f"1 fraud per {legit_count // fraud_count} legit txns"
    },
    "flagged_fraud"  : {
        "isFlaggedFraud_count"    : flagged_count,
        "fraud_missed_by_rules"   : fraud_count - flagged_count
    },
    "transaction_types" : {
        "all_types"         : df["type"].unique().tolist(),
        "fraud_only_types"  : fraud_types,
        "non_fraud_types"   : [t for t in df["type"].unique() if t not in fraud_types]
    },
    "amount_insights" : {
        "fraud_mean_amount" : round(fraud_mean_amt, 2),
        "legit_mean_amount" : round(legit_mean_amt, 2),
        "fraud_amount_ratio": round(fraud_mean_amt / legit_mean_amt, 2)
    },
    "balance_insights" : {
        "zero_newbalanceOrig_fraud_count" : zero_bal_fraud,
        "zero_newbalanceOrig_fraud_pct"   : round(zero_bal_pct, 2),
        "raw_balance_cols_usable"         : False,
        "reason"                          : "Balances are manipulated in fraud rows per dataset note"
    },
    "temporal_insights" : {
        "step_min"          : int(df["step"].min()),
        "step_max"          : int(df["step"].max()),
        "simulation_days"   : round(df["step"].max() / 24, 1),
        "late_period_fraud_rate_pct" : round(
            df[df["time_period"] == "Late"]["isFraud"].mean() * 100, 4
        )
    },
    "feature_engineering_hints" : [
        "is_transfer_or_cashout — only these types have fraud",
        "is_zero_newbalanceOrig — 98% of fraud drains origin to 0",
        "amount_log — amount is heavily right-skewed, log-transform needed",
        "step_mod_24 — hour-of-day signal from step column",
        "is_late_period — Late period has 10x higher fraud rate",
        "orig_balance_diff — deviation from expected balance equation",
        "dest_balance_diff — deviation from expected balance equation"
    ]
}

# Save to data/processed/
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "eda_summary.json"
with open(output_path, "w") as f:
    json.dump(summary, f, indent=4)

print("=" * 60)
print("EDA SUMMARY SAVED")
print("=" * 60)
print(f"   Saved to : {output_path.resolve()}")
print(f"\n   Key findings:")
for hint in summary["feature_engineering_hints"]:
    print(f"   ✅  {hint}")

print(f"\n   ✅ Day 2 EDA complete — notebook 01_eda.ipynb done")


EDA SUMMARY SAVED
   Saved to : /home/minipekka/Hackathons/USM VHack 2026/nuroguard/data/processed/eda_summary.json

   Key findings:
   ✅  is_transfer_or_cashout — only these types have fraud
   ✅  is_zero_newbalanceOrig — 98% of fraud drains origin to 0
   ✅  amount_log — amount is heavily right-skewed, log-transform needed
   ✅  step_mod_24 — hour-of-day signal from step column
   ✅  is_late_period — Late period has 10x higher fraud rate
   ✅  orig_balance_diff — deviation from expected balance equation
   ✅  dest_balance_diff — deviation from expected balance equation

   ✅ Day 2 EDA complete — notebook 01_eda.ipynb done
